[Step 10 - Minimal RAG pipeline]

> **MLCourse - Agentic AI - Basic RAG**

> Stage in the capstone: THIS IS the generation core the capstone wraps memory around.

Retrieval-Augmented Generation answers questions **from YOUR documents** instead of
from the model's frozen training memory: fetch a few relevant chunks, paste them into
a rule-bound prompt, and let the model read-and-cite rather than recall-and-guess.
This notebook builds the smallest complete RAG pipeline end to end - the exact
skeleton the Step 12 capstone grows into.

### What you will learn

1. Recreate the shared corpus assets defensively (download-once alice.txt, chunks with ids, persisted Chroma index).
2. Format retrieved chunks with `[id]` tags so the model can cite its evidence.
3. Wire the classic LCEL RAG chain: dict prelude -> prompt -> llm -> string parser.
4. Compare BOTH context-filling idioms: dict parallel and `RunnablePassthrough.assign`.
5. Run three real Alice questions plus an out-of-corpus refusal test.
6. Grade relevance BY EYE with a printed checklist (metrics-lite).

### Sections

1. Setup - imports, track discovery, data dir, env keys, notebook magic guard
2. Corpus - download once, chunk, tag chunk ids
3. Index - keyless MiniLM embeddings into a persisted Chroma store
4. Model - guarded Ollama llama3.2 with a clearly-labelled offline stub
5. Prompt + chain - answer ONLY from provided context, cite chunk ids
6. Three real questions, each printed with its retrieved evidence
7. Failure mode - an absurd out-of-corpus question
8. Optional Groq comparison + the manual eyeball checklist
9. Summary

In [1]:
# --- Section 1: setup --------------------------------------------------------
# Every notebook in this track opens with the same four chores:
#   imports, find the track folder, resolve DATA, load .env keys.
# The matplotlib "magic" guard only fires inside Jupyter; in plain Python,
# get_ipython() does not exist and the try/except quietly moves on.
from pathlib import Path          # paths that behave on Windows / mac / linux
import os                         # read environment variables (API keys)
import re                         # regex: pull cited [id] numbers out of answers
import urllib.request             # one-time download of alice.txt

def _find_track(start_dir):
    """Climb parent folders until we find (or reach) the dir named 03_agentic_ai."""
    here = Path(start_dir).resolve()
    for candidate in (here, *here.parents):        # this dir, then every ancestor
        if candidate.name == "03_agentic_ai":      # we are INSIDE the track already
            return candidate
        if (candidate / "03_agentic_ai").is_dir(): # track sits below this dir
            return candidate / "03_agentic_ai"
    raise FileNotFoundError("Could not locate the 03_agentic_ai track near %s" % here)

TRACK = _find_track(Path.cwd())    # track root: holds data/, .env/, module folders
DATA = TRACK / "data"              # shared corpus + indexes live here
DATA.mkdir(parents=True, exist_ok=True)

from dotenv import load_dotenv     # reads .env into os.environ - never hardcode keys
load_dotenv(TRACK / ".env", override=False)   # canonical location per track README
load_dotenv(override=False)                   # also honor any .env near the cwd

try:                               # notebook-only convenience for later plots
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:                  # plain-script run: nothing to do
    pass

print("track:", TRACK)
print("data :", DATA)

track: D:\projects\python\MLCourse\03_agentic_ai
data : D:\projects\python\MLCourse\03_agentic_ai\data


In [2]:
# --- Section 2a: get the corpus ----------------------------------------------
# WHY download-once: Gutenberg is a shared resource and networks fail; after the
# first successful download every rerun uses the local copy instantly.
ALICE_PATH = DATA / "alice.txt"
ALICE_URL = "https://www.gutenberg.org/files/11/11-0.txt"
if not ALICE_PATH.exists():
    print("first run: downloading", ALICE_URL)
    urllib.request.urlretrieve(ALICE_URL, ALICE_PATH)   # blocking, ~170 KB
print("alice.txt ready:", ALICE_PATH.stat().st_size, "bytes")

from langchain_community.document_loaders import TextLoader
raw_doc = TextLoader(str(ALICE_PATH), encoding="utf-8").load()[0]  # ONE huge Document
print("loaded %d characters as a single Document" % len(raw_doc.page_content))

# Chunk it with the track-standard recursive splitter: 500-char chunks, 100 overlap.
# Recursive tries paragraph -> line -> word boundaries, so dialogue stays readable.
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
# split_documents expects a LIST of Documents - normalize defensively:
raw_docs = raw_doc if isinstance(raw_doc, list) else [raw_doc]
chunks = splitter.split_documents(raw_docs)

for i, ch in enumerate(chunks):            # stamp a stable integer id on every chunk
    ch.metadata["id"] = i                  # the prompt will REQUIRE citing these ids

sizes = [len(c.page_content) for c in chunks]
print("chunks: %d | size min/avg/max = %d/%d/%d chars"
      % (len(chunks), min(sizes), sum(sizes) // len(sizes), max(sizes)))

alice.txt ready: 151191 bytes


C:\Users\Thoyajaksha Kashyap\AppData\Local\Temp\ipykernel_56348\3103869174.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


loaded 144696 characters as a single Document
chunks: 411 | size min/avg/max = 26/374/498 chars


In [3]:
# --- Section 2b: embeddings + persisted vector store -------------------------
# Embeddings are UNGUARDED on purpose: all-MiniLM-L6-v2 runs keyless on your CPU
# via sentence-transformers. First ever run downloads ~90 MB, then it is offline.
from langchain_huggingface import HuggingFaceEmbeddings
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

from langchain_chroma import Chroma
CHROMA_DIR = DATA / "chroma_capstone"      # shared with modules 08-12
COLLECTION = "alice_rag"

def open_or_build_vectorstore(chunk_list):
    """Reuse an existing on-disk index; embed+store ONLY when none exists yet.

    Defensive recreate pattern: blindly calling add_documents every rerun would
    duplicate every vector (and silently skew retrieval). Opening first costs
    nothing; rebuilding happens exactly once, on the very first run.
    """
    if CHROMA_DIR.exists():                                   # previous run left an index?
        try:
            vs = Chroma(collection_name=COLLECTION, embedding_function=embeddings,
                        persist_directory=str(CHROMA_DIR))
            if len(vs.get()["ids"]) > 0:                      # non-empty: reuse as-is
                return vs
        except Exception as exc:                              # corrupt/partial index:
            print("reopening failed (%s); rebuilding from scratch" % type(exc).__name__)
    vs = Chroma(collection_name=COLLECTION, embedding_function=embeddings,
                persist_directory=str(CHROMA_DIR))
    vs.add_documents(chunk_list)      # first run: embed all chunks (takes a minute)
    return vs

vectorstore = open_or_build_vectorstore(chunks)
print("index '%s' at %s holds %d vectors"
      % (COLLECTION, CHROMA_DIR.name, len(vectorstore.get()["ids"])))

# Retriever = vectorstore plus a search policy. k=3 keeps prompts small and forces
# precision; the capstone will widen to MMR k=4/fetch_k=16 and justify the change.
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

index 'alice_rag' at chroma_capstone holds 411 vectors


In [4]:
# --- Section 3: guarded chat model with an offline stub -----------------------
# Provider rules of this track:
#   PRIMARY  : Ollama llama3.2 (free, local) - probe once, degrade gracefully.
#   SECONDARY: Groq llama-3.3-70b-versatile - only when GROQ_API_KEY exists.
# If Ollama is unreachable we swap in a RunnableLambda stub that assembles a canned
# answer FROM THE RETRIEVED CONTEXT so the PIPELINE mechanics still run anywhere.
from langchain_ollama import ChatOllama
PRIMARY_MODEL = "llama3.2"
base_llm = ChatOllama(model=PRIMARY_MODEL, temperature=0)   # temp 0: factual answers

LLM_LIVE = False
try:
    base_llm.invoke("Reply with the single word: pong")     # tiny reachability probe
    LLM_LIVE = True
    print("primary chat model online:", PRIMARY_MODEL, "(local Ollama)")
except Exception as exc:
    print("[demo skipped] install/start Ollama and run: ollama pull llama3.2")
    print("   detail: %s: %s" % (type(exc).__name__, exc))

def fake_answer(prompt_value):
    """OFFLINE STUB so the PIPELINE mechanics run anywhere; swap llm back for real answers.

    Receives whatever the previous LCEL step produced (usually a PromptValue whose
    messages contain our rules + question + [id]-tagged context), harvests the
    cited chunk ids and a snippet, and returns a deterministic string.
    """
    try:
        blob = "\n".join(getattr(m, "content", "") for m in prompt_value.to_messages())
    except Exception:
        blob = str(prompt_value)                            # last-resort text form
    ids = list(dict.fromkeys(re.findall(r"\[(\d+)\]", blob)))  # unique, order kept
    tail = blob.rsplit("Context:", 1)[-1]
    snippet = " ".join(tail.split())[:200]
    return ("[offline stub] citing chunks [%s]; closest retrieved text starts: \"%s\" "
            "(offline stub so the PIPELINE mechanics run anywhere; swap llm back for real answers)"
            % (",".join(ids) or "?", snippet))

llm = base_llm if LLM_LIVE else None
if llm is None:
    from langchain_core.runnables import RunnableLambda
    llm = RunnableLambda(fake_answer)                       # same interface, no GPU needed
    print(">> running with the OFFLINE STUB model for this session")

[demo skipped] install/start Ollama and run: ollama pull llama3.2
   detail: ConnectError: [WinError 10061] No connection could be made because the target machine actively refused it
>> running with the OFFLINE STUB model for this session


In [5]:
# --- Section 4: the RAG prompt and BOTH wiring idioms -------------------------
# Contract with the model, in three rules: (1) use ONLY the provided context,
# (2) cite the chunk ids you used like [12], (3) say a fixed refusal sentence when
# the context lacks the answer. Rule 3 is what turns hallucination into honesty.
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a precise research assistant for ONE book: Alice's Adventures in Wonderland.\n"
     "Rules:\n"
     "1. Answer ONLY from the provided context.\n"
     "2. Cite the chunk ids you used, in square brackets like [12].\n"
     "3. If the context does not contain the answer, reply exactly: "
     "'I cannot find that in the provided excerpts.'"),
    ("human", "Question: {question}\n\nContext:\n{context}"),
])

LAST_DOCS = []                       # stash of the docs used by the latest call,
                                     # so we can print evidence right after answers
def fetch_context(inputs: dict) -> str:
    """Retriever call wrapped as a runnable step: {'question': str} -> tagged text."""
    hits = retriever.invoke(inputs["question"])
    LAST_DOCS.clear()
    LAST_DOCS.extend(hits)
    return "\n\n".join("[%s] %s" % (d.metadata.get("id", "?"), d.page_content.strip())
                       for d in hits)

# IDIOM A - dict parallel prelude (what we will use everywhere):
# both branches receive {"question": ...}; itemgetter copies the question through.
prelude = {
    "context": RunnableLambda(fetch_context),
    "question": itemgetter("question"),
}
rag_chain = prelude | RAG_PROMPT | llm | StrOutputParser()

# IDIOM B - RunnablePassthrough.assign: keeps existing keys AND adds 'context'.
rag_chain_assign = (
    RunnablePassthrough.assign(context=RunnableLambda(fetch_context))
    | RAG_PROMPT | llm | StrOutputParser()
)
print("chains built. Idiom A = dict prelude, Idiom B = assign - same contract.")

chains built. Idiom A = dict prelude, Idiom B = assign - same contract.


In [6]:
# --- Section 5: three real questions with evidence ----------------------------
# For each question we print: the answer, the retrieved chunk previews, and an
# automatic grounding check (do the cited ids exist among the retrieved chunks?).
QUESTIONS = [
    "Who does Alice have a conversation with about a race?",       # Dodo / Caucus-race
    "What does the White Rabbit mistake Alice for?",               # Mary Ann, his housemaid
    "What does the Queen of Hearts keep shouting when angry?",     # Off with their heads!
]

def ask_and_show(chain, question: str) -> str:
    """One audited turn: guarded invoke + evidence preview + grounding check."""
    LAST_DOCS.clear()
    print("=" * 72)
    print("Q:", question)
    try:
        answer = chain.invoke({"question": question})
    except Exception as exc:
        print("[demo skipped] install/start Ollama and run: ollama pull llama3.2")
        print("   detail: %s: %s" % (type(exc).__name__, exc))
        return ""
    print("\nA:", answer.strip()[:600])
    print("\nretrieved evidence:")
    for d in LAST_DOCS:
        preview = " ".join(d.page_content.split())[:100]
        print("   [%s] %s..." % (d.metadata.get("id", "?"), preview))
    cited = {int(x) for x in re.findall(r"\[(\d+)\]", answer)}
    available = {int(d.metadata.get("id", -1)) for d in LAST_DOCS}
    verdict = "yes" if cited and cited.issubset(available) else \
              "no citations found" if not cited else "cited id NOT in retrieved set!"
    print("\ncitations grounded in retrieved chunks:", verdict)
    return answer

answers = {}
for q in QUESTIONS:
    answers[q] = ask_and_show(rag_chain, q)

# Same question through Idiom B: outputs should agree (identical prompt bytes).
alt = rag_chain_assign.invoke({"question": QUESTIONS[0]})
same = (str(alt).strip() == str(answers[QUESTIONS[0]]).strip())
print("=" * 72)
print("assign-idiom output matches dict-prelude output:", same,
      "- both idioms feed the identical prompt.")

Q: Who does Alice have a conversation with about a race?

A: [offline stub] citing chunks [12,218,180,217]; closest retrieved text starts: "[218] “You should learn not to make personal remarks,” Alice said with some severity; “it’s very rude.” The Hatter opened his eyes very wide on hearing this; but all he _said_ was, “Why is a raven lik" (offline stub so the PIPELINE mechanics run anywhere; swap llm back for real answers)

retrieved evidence:
   [218] “You should learn not to make personal remarks,” Alice said with some severity; “it’s very rude.” Th...
   [180] “Please, then,” said Alice, “how am I to get in?”...
   [217] “There isn’t any,” said the March Hare. “Then it wasn’t very civil of you to offer it,” said Alice a...

citations grounded in retrieved chunks: yes
Q: What does the White Rabbit mistake Alice for?

A: [offline stub] citing chunks [12,100,376,42]; closest retrieved text starts: "[100] Very soon the Rabbit noticed Alice, as she went hunting about, and called out to 

In [7]:
# --- Section 6: failure mode - absurd out-of-corpus question ------------------
# The prompt FORBIDS outside knowledge, so a well-behaved model must answer with
# the fixed refusal sentence. This is your anti-hallucination smoke test.
WEIRD_Q = "Who won the football World Cup final in 2022?"
weird_answer = ask_and_show(rag_chain, WEIRD_Q)

print("-" * 72)
if LLM_LIVE:
    refused = "cannot find that" in weird_answer.lower()
    print("live-model refusal observed:", refused,
          "-> the system rules, not the retriever, produced this behavior")
else:
    print("(offline stub cannot refuse - refusals are MODEL behavior; rerun with Ollama)")
print("takeaway: ALWAYS include an out-of-corpus question in your RAG test plan.")

Q: Who won the football World Cup final in 2022?

A: [offline stub] citing chunks [12,82,83,371]; closest retrieved text starts: "[82] again, the Dodo suddenly called out “The race is over!” and they all crowded round it, panting, and asking, “But who has won?” [83] This question the Dodo could not answer without a great deal of" (offline stub so the PIPELINE mechanics run anywhere; swap llm back for real answers)

retrieved evidence:
   [82] again, the Dodo suddenly called out “The race is over!” and they all crowded round it, panting, and ...
   [83] This question the Dodo could not answer without a great deal of thought, and it sat for a long time ...
   [371] “Then you may _sit_ down,” the King replied. Here the other guinea-pig cheered, and was suppressed. ...

citations grounded in retrieved chunks: yes
------------------------------------------------------------------------
(offline stub cannot refuse - refusals are MODEL behavior; rerun with Ollama)
takeaway: ALWAYS include an 

In [8]:
# --- Section 7: optional Groq comparison + metrics-lite checklist -------------
# SECONDARY provider behind a key guard: without GROQ_API_KEY this prints a skip
# message and everything above has already proven the pipeline locally.
groq_llm = None
if os.getenv("GROQ_API_KEY"):
    from langchain_groq import ChatGroq
    groq_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    groq_chain = prelude | RAG_PROMPT | groq_llm | StrOutputParser()
    try:
        ga = groq_chain.invoke({"question": QUESTIONS[0]})
        print("Groq llama-3.3-70b-versatile on Q1:\n", ga.strip()[:500])
    except Exception as exc:
        print("[groq skipped] call failed (%s: %s)" % (type(exc).__name__, exc))
        print("   free-tier HTTP 429 rate limits usually clear after a short wait")
else:
    print("[groq skipped] GROQ_API_KEY not set in 03_agentic_ai/.env - comparison skipped")
    print("   which is fine: the local path above IS the lesson")

# Manual relevance grading - the honest first metric of any RAG project.
CHECKLIST = [
    "Q1 cites a chunk mentioning the race / Caucus-race (the Dodo proposes it)",
    "Q2 cites a chunk containing 'Mary Ann' (the Rabbit's mistake)",
    "Q3 cites a chunk containing 'Off with' (the Queen's catchphrase)",
    "every cited [id] appears among the printed evidence previews",
    "World Cup question produced a refusal, not invented facts (needs live model)",
]
print("\n=== METRICS-LITE: tick these by eye against the outputs above ===")
for item in CHECKLIST:
    print("   [ ]", item)
print("eyeballing retrieval quality takes 60 seconds and catches most regressions.")

[groq skipped] GROQ_API_KEY not set in 03_agentic_ai/.env - comparison skipped
   which is fine: the local path above IS the lesson

=== METRICS-LITE: tick these by eye against the outputs above ===
   [ ] Q1 cites a chunk mentioning the race / Caucus-race (the Dodo proposes it)
   [ ] Q2 cites a chunk containing 'Mary Ann' (the Rabbit's mistake)
   [ ] Q3 cites a chunk containing 'Off with' (the Queen's catchphrase)
   [ ] every cited [id] appears among the printed evidence previews
   [ ] World Cup question produced a refusal, not invented facts (needs live model)
eyeballing retrieval quality takes 60 seconds and catches most regressions.


## Summary

- One sentence: retrieve 3 relevant `[id]`-tagged chunks, paste them under strict
  rules into the prompt, generate a cited answer - or refuse.
- The chain is five movable parts: retriever step, format step, prompt, model,
  parser. Dict-prelude and `RunnablePassthrough.assign` are interchangeable ways
  to fill `{context}` while copying `{question}` through.
- Guards keep the notebook green anywhere: Ollama missing -> labelled offline
  stub; no Groq key -> skip message; index exists -> reuse, else build once.
- Quality checks you can run tonight: grounding check per answer, an
  out-of-corpus refusal probe, and the 60-second eyeball checklist.

Next: Step 11 gives this generation core a per-session memory.